# Factor IO — chat-with-your-data demo backend

Runs **Qwen3.8-Flash-Next** on the [Strata](https://github.com/Niko1221/Strata) engine on this Colab VM, plus the demo gateway and a public HTTPS tunnel. At the end it prints a **connect link** for the customer.

1. **Runtime → Change runtime type → G4** (a G4 has about 176 GB of RAM and a 96 GB GPU, which fit the NVFP4 model whole).
2. **Runtime → Run all.** First boot downloads the model (NVFP4 is 128 GB), so expect a wait before the link appears.
3. Send the link. Keep this tab open: the demo stops when the runtime stops, and customers' files go with it.
4. When you are done, tick **STOP_DEMO** in the last cell and run that cell. Run all leaves it unticked, so it never stops the demo by itself.

Nothing here runs on your own computer.

In [ ]:
#@title Settings
MODEL = "auto"  #@param ["auto", "nvfp4", "iq3_s", "iq3_xxs", "iq2_xs", "q2_0"]
CONTEXT = "64k"  #@param ["32k", "64k", "128k"]
PARALLEL = 2  #@param {type:"slider", min:1, max:4, step:1}
#@markdown `auto` picks NVFP4 when there is ≥100 GB RAM and ≥150 GB free disk, else IQ3_S, else Q2_0.
SOURCE = "https://studio.factor-io.com/demo/chat/backend/"  #@param {type:"string"}
PAGE_URL = "https://studio.factor-io.com/demo/chat/"  #@param {type:"string"}

In [ ]:
#@title Check the GPU and fetch the backend files
import os, subprocess, urllib.request
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], capture_output=True, text=True).stdout or "No GPU: change the runtime type to G4.")
print(subprocess.run("free -g | head -2; df -h /content | tail -1", shell=True, capture_output=True, text=True).stdout)
WORK = "/content/factor-chat"
os.makedirs(WORK, exist_ok=True)
for name in ["colab_start.sh", "gateway.py", "rag.py", "requirements.txt"]:
    urllib.request.urlretrieve(SOURCE + name, os.path.join(WORK, name))
    print("fetched", name)

In [ ]:
#@title Start Strata, the gateway and the tunnel (long on first boot)
import subprocess, sys
env = dict(os.environ, MODEL=MODEL, CONTEXT=CONTEXT, PARALLEL=str(PARALLEL), PAGE_URL=PAGE_URL, KEEPALIVE="0")
p = subprocess.Popen(["bash", "colab_start.sh"], cwd=WORK, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    sys.stdout.write(line)
if p.wait() != 0:
    raise SystemExit("Start failed: read the lines above, or run the Status cell.")

In [ ]:
#@title Connect link
import json, html
from IPython.display import HTML, display
s = json.load(open(f"{WORK}/logs/status.json"))
if s.get("state") != "ready":
    print("Not ready:", s.get("state"), s.get("message"))
else:
    link = html.escape(s["link"])
    display(HTML(f"""<div style='font:15px system-ui;padding:14px;border:1px solid #888;border-radius:10px'>
      <b>Ready</b> — {html.escape(s.get('gpu',''))}, Qwen3.8-Flash-Next {html.escape(s.get('size',''))}<br><br>
      <a href='{link}' target='_blank'>{link}</a><br><br>
      <button onclick="navigator.clipboard.writeText('{link}');this.textContent='Copied'">Copy link</button>
      <div style='margin-top:8px;opacity:.7'>Backend URL: {html.escape(s['tunnel'])}</div></div>"""))

In [ ]:
#@title Status (run any time)
import os, urllib.request
STRATA_PORT = os.environ.get("STRATA_PORT", "8081")  # colab_start.sh's default; Colab itself holds 8080
GATEWAY_PORT = os.environ.get("GATEWAY_PORT", "8000")
def get(url):
    try:
        return urllib.request.urlopen(url, timeout=5).status
    except Exception as e:
        return type(e).__name__
print(f"strata  :{STRATA_PORT}/health  ->", get(f"http://127.0.0.1:{STRATA_PORT}/health"))
print(f"gateway :{GATEWAY_PORT}/api/ping ->", get(f"http://127.0.0.1:{GATEWAY_PORT}/api/ping"))
print(open(f"{WORK}/logs/status.json").read())
print(subprocess.run(f"nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv; free -g | head -2; tail -n 15 {WORK}/logs/gateway.log", shell=True, capture_output=True, text=True).stdout)

In [ ]:
#@title Stop the demo and release the GPU (ends billing; the link dies)
STOP_DEMO = False  #@param {type:"boolean"}
#@markdown Tick the box, then run this cell. Run all leaves it unticked, so the demo keeps running.
if STOP_DEMO:
    from google.colab import runtime
    runtime.unassign()
else:
    print("The demo is running. To stop it, tick STOP_DEMO above and run this cell.")